# **Modelos Referenciales**
### Proyecto Hito 1
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

## Índice

>[0- Instalación de librerías](#0--instalación-de-librerías)

>[1- Carga de datos](#1--carga-de-datos)

>[2- User KNN](#2--user-knn)

>[3- Item KNN](#3--item-knn)

>[4- Most Popular](#4--most-popular)

>[5- Random](#5--random)

## 0- Instalación de librerías

In [1]:
# !pip uninstall -y numpy
# !pip install numpy==1.26

In [2]:
# !pip install scikit-surprise --no-build-isolation --no-deps

In [3]:
# pip install pandas

## 1- Carga de datos

In [ ]:
import surprise
import numpy as np
import pandas as pd
from collections import defaultdict
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy
import random

In [5]:
df = pd.read_csv('video_game_reviews.csv')

In [6]:
np.random.seed(42)

In [7]:
# añadir un id de usuario a cada fila del dataframe entre 1 y 3000 de forma aleatoria
df['user_id'] = np.random.randint(1, 3001, size=len(df))

In [8]:
print(f"Juegos unicos: {df['Game Title'].unique()}")

Juegos unicos: ['Grand Theft Auto V' 'The Sims 4' 'Minecraft' 'Bioshock Infinite'
 'Half-Life: Alyx' 'Sid Meier’s Civilization VI' 'Just Dance 2024'
 '1000-Piece Puzzle' 'Spelunky 2' 'Street Fighter V' 'Fall Guys'
 'Rocket League' 'The Elder Scrolls V: Skyrim' 'Among Us' 'Stardew Valley'
 'Call of Duty: Modern Warfare 2'
 'The Legend of Zelda: Breath of the Wild' 'Tekken 7'
 'Pillars of Eternity II: Deadfire' 'Animal Crossing: New Horizons'
 'Hades' 'Mario Kart 8 Deluxe' 'Overwatch 2' 'Fortnite'
 'Pokémon Scarlet & Violet' 'Hitman 3' 'Tomb Raider (2013)'
 'Halo Infinite' 'Super Smash Bros. Ultimate' 'Kingdom Hearts III'
 'League of Legends' 'The Witcher 3: Wild Hunt' 'FIFA 24'
 'Ghost of Tsushima' 'Cuphead' 'Red Dead Redemption 2' 'Portal 2' 'Tetris'
 'Counter-Strike: Global Offensive' 'Super Mario Odyssey']


In [9]:
print(f"Juegos unicos: {df['Game Title'].nunique()}")

Juegos unicos: 40


In [10]:
# a cada titulo de juego le asignamos un id numerico
df['item_id'] = df['Game Title'].astype('category').cat.codes

In [11]:
# lista con la info de las pelis, en forma de diccionario item_id: titulo

info_pelis = dict(zip(df['item_id'], df['Game Title']))

In [12]:
print(info_pelis)

{11: 'Grand Theft Auto V', 37: 'The Sims 4', 20: 'Minecraft', 3: 'Bioshock Infinite', 13: 'Half-Life: Alyx', 27: 'Sid Meier’s Civilization VI', 16: 'Just Dance 2024', 0: '1000-Piece Puzzle', 28: 'Spelunky 2', 30: 'Street Fighter V', 8: 'Fall Guys', 26: 'Rocket League', 35: 'The Elder Scrolls V: Skyrim', 1: 'Among Us', 29: 'Stardew Valley', 4: 'Call of Duty: Modern Warfare 2', 36: 'The Legend of Zelda: Breath of the Wild', 33: 'Tekken 7', 22: 'Pillars of Eternity II: Deadfire', 2: 'Animal Crossing: New Horizons', 12: 'Hades', 19: 'Mario Kart 8 Deluxe', 21: 'Overwatch 2', 9: 'Fortnite', 23: 'Pokémon Scarlet & Violet', 15: 'Hitman 3', 39: 'Tomb Raider (2013)', 14: 'Halo Infinite', 32: 'Super Smash Bros. Ultimate', 17: 'Kingdom Hearts III', 18: 'League of Legends', 38: 'The Witcher 3: Wild Hunt', 7: 'FIFA 24', 10: 'Ghost of Tsushima', 6: 'Cuphead', 25: 'Red Dead Redemption 2', 24: 'Portal 2', 34: 'Tetris', 5: 'Counter-Strike: Global Offensive', 31: 'Super Mario Odyssey'}


In [13]:
# lista de columnas actuales
cols = df.columns.tolist()
new_order = ['user_id', 'item_id'] + [c for c in cols if c not in ['user_id', 'item_id']]
df = df[new_order]


In [14]:
df.to_csv('video_game_reviews_with_userid.csv', index=False)

In [15]:
# contar cuantas filas hay por cada user_id
print(df['user_id'].value_counts())

user_id
2839    32
2135    30
2774    30
948     30
993     29
        ..
544      5
424      5
2465     5
1581     5
2050     5
Name: count, Length: 3000, dtype: int64


In [16]:
# contar usuarios unicos
print(f"Numero de usuarios unicos: {df['user_id'].nunique()}")

Numero de usuarios unicos: 3000


In [17]:
df = pd.read_csv('video_game_reviews_with_userid.csv', sep=',')

df = df[['user_id', 'item_id', 'User Rating']]

In [18]:
df.loc[:, 'User Rating'] = round(df.loc[:, 'User Rating'] / 10, 2)

In [19]:
df

,user_id,item_id,User Rating
0,861,11,3.64
1,1295,37,3.83
2,1131,20,2.68
3,1096,3,3.84
4,1639,13,3.01
...,...,...,...
47769,1293,20,4.16
47770,2485,36,2.42
47771,2675,2,2.67
47772,2599,36,2.25


In [20]:
df.to_csv('video_game_reviews_with_userid_clean.csv', index=False)

In [ ]:
reader = Reader(line_format='user item rating', sep=',', rating_scale=(1,5), skip_lines=1)
data = Dataset.load_from_file('video_game_reviews_with_userid_clean.csv', reader=reader)

trainset, testset = train_test_split(data, test_size=0.2)

print("Usuarios:", trainset.n_users, "Items:", trainset.n_items, "Test size:", len(testset))

Usuarios: 3000 Items: 40 Test size: 9555


## 2- User KNN

Se busca el mejor valor de 'k' entre un conjunto predefinido 'k_values'. Se prueba usando la correlación de Pearson y la similitud de coseno como medidas de similtud. Finalmente, se elige el valor de 'k' y la métrica de similitud con los que se obtiene menor valor de RMSE:

In [ ]:
#Valores de k que se probarán:
k_values = [5, 10, 20, 30, 50, 70, 100]
rmse_values_user_knn = []
sim_options = ["cosine", "pearson"]

for option in sim_options:
  for k in k_values:
    myUserKnn = surprise.KNNBasic(k=k, sim_options={'name': option, 'user_based': True})
    myUserKnn.fit(trainset)
    predictions = myUserKnn.test(testset)
    rmse_values_user_knn.append([option, k, accuracy.rmse(predictions, verbose=False)])

best_cosine = None
best_pearson = None

for sim, k, rmse in rmse_values_user_knn:
    if sim == "cosine":
        if best_cosine is None or rmse < best_cosine[2]:
            best_cosine = [sim, k, rmse]
    elif sim == "pearson":
        if best_pearson is None or rmse < best_pearson[2]:
            best_pearson = [sim, k, rmse]

print("Mejor cosine :", best_cosine)
print("Mejor pearson:", best_pearson)

Para realizar la recomendación, se define el modelo User KNN con similitud de coseno y k = 50, que es la combinación que alcanzó menor RMSE:

In [24]:
myUserKnn = surprise.KNNBasic(k=50, sim_options={'name': 'cosine', 'user_based': True})

In [25]:
myUserKnn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


Como ejemplo, se prueba predecir el rating que el usuario de ID = 457 daría al videojuego de ID = 0:

In [27]:
myUserKnn.predict("457", "0")

Prediction(uid='457', iid='0', r_ui=None, est=2.98800067433686, details={'actual_k': 50, 'was_impossible': False})

Se realizan las predicciones a partir del antitest set:

In [28]:
a_testset = trainset.build_anti_testset()
predictions = myUserKnn.test(a_testset)

Función para obtener lista de recomendación top N:

In [29]:
def get_top_n(predictions, n=10):
    """Devuelve las N-mejores recomendaciones para cada usuario de un set de predicción.

    Args:
        predictions(lista de objetos Prediction): La lista de predicción obtenida del método test.
        n(int): El número de recomendaciónes por usuario

    Returns:
    Un diccionario donde las llaves son ids de usuario y los valores son listas de tuplas:
        [(item id, rating estimation), ...] de tamaño n.
    """

    # First map the predictions to each user.
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))

    # Then sort the predictions for each user and retrieve the k highest ones.
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]

    return top_n

Se obtiene la lista de recomendación top 10 para cada usuario y s emuestra como ejemplo el caso del con ID = 457:

In [30]:
top_n = get_top_n(predictions, n=10)
print(top_n["457"])

[('10', 3.1748000000000003), ('21', 3.1139993529874572), ('32', 3.1061999999999994), ('8', 3.0872057284510412), ('7', 3.0649999999999995), ('28', 3.0465889615866675), ('9', 3.0417983263205843), ('12', 3.0225911471351985), ('17', 3.0102163238227), ('37', 3.005397982139244)]


Se muestran los títulos de las películas recomendadas al usuario ID = 457 junto a su rating predicho:

In [ ]:
for item_id, rating in top_n["457"]:
    print(f"{info_pelis[int(item_id)]}: {rating:.2f}")

Ghost of Tsushima: 3.17
Overwatch 2: 3.11
Super Smash Bros. Ultimate: 3.11
Fall Guys: 3.09
FIFA 24: 3.06
Spelunky 2: 3.05
Fortnite: 3.04
Hades: 3.02
Kingdom Hearts III: 3.01
The Sims 4: 3.01


## 3- Item KNN

Se busca el mejor valor de K entre un conjunto predefinido 'k_values'. Se prueba usando la correlación de Pearson y la similitud de coseno como medidas de similtud. Finalmente, se elige el valor de K y la métrica de similitud con los que se obtiene menor valor de RMSE:

In [ ]:
#Valores de k que se probarán:
k_values = [5, 10, 20, 30, 50, 70, 100]
rmse_values_user_knn = []
sim_options = ["cosine", "pearson"]

for option in sim_options:
  for k in k_values:
    myUserKnn = surprise.KNNBasic(k=k, sim_options={'name': option, 'user_based': False})
    myUserKnn.fit(trainset)
    predictions = myUserKnn.test(testset)
    rmse_values_user_knn.append([option, k, accuracy.rmse(predictions, verbose=False)])

best_cosine = None
best_pearson = None

for sim, k, rmse in rmse_values_user_knn:
    if sim == "cosine":
        if best_cosine is None or rmse < best_cosine[2]:
            best_cosine = [sim, k, rmse]
    elif sim == "pearson":
        if best_pearson is None or rmse < best_pearson[2]:
            best_pearson = [sim, k, rmse]

print("Mejor cosine :", best_cosine)
print("Mejor pearson:", best_pearson)

Para realizar la recomendación, se define el modelo Item KNN con similitud de coseno y k = 30, que es la combinación que alcanzó menor RMSE:

In [33]:
myItemKnn = surprise.KNNBasic(k=30, sim_options={'name': 'cosine', 'user_based': False})

In [34]:
myItemKnn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


Como ejemplo, se prueba predecir el rating que el usuario de ID = 457 daría al videojuego de ID = 0:

In [ ]:
myItemKnn.predict("457", "0")

Se realizan las predicciones a partir del antitest set definido en la sección anterior:

In [35]:
predictions_item = myItemKnn.test(a_testset)

Se obtiene la lista de recomendación top 10 para cada usuario (usando get_top_n() definida arriba) y se muestra como ejemplo el caso del con ID = 457:

In [36]:
top_n_item = get_top_n(predictions_item, n=10)
print(top_n_item["457"])

[('0', 2.713754550867083), ('3', 2.7135192760190727), ('39', 2.7133298523367793), ('9', 2.713275521038186), ('31', 2.713203696480411), ('38', 2.713099835046572), ('27', 2.7130531655317864), ('16', 2.71299967697946), ('1', 2.7129800505046813), ('18', 2.712927813150408)]


Se muestran los títulos de las películas recomendadas al usuario ID = 457 junto a su rating predicho:

In [37]:
for item_id, rating in top_n_item["457"]:
    print(f"{info_pelis[int(item_id)]}: {rating:.8f}")

1000-Piece Puzzle: 2.71375455
Bioshock Infinite: 2.71351928
Tomb Raider (2013): 2.71332985
Fortnite: 2.71327552
Super Mario Odyssey: 2.71320370
The Witcher 3: Wild Hunt: 2.71309984
Sid Meier’s Civilization VI: 2.71305317
Just Dance 2024: 2.71299968
Among Us: 2.71298005
League of Legends: 2.71292781


## 4- Most Popular

Se realiza el procedimiento para recomendar a cada usuario los 10 videojuegos más populares que no haya visto. Para ello, se calcula para cada videojuego un 'score' de popularidad, que es la suma de todos los rating con el que los usuarios le han evaluado. Luego, los videojuegos más populares serán aquellos con el score más alto. Se define una función que realiza la recomendación top N para cada usuario:

In [ ]:
def get_top_n_most_popular(trainset, n=10):

    #Se calcula el score de popularidad de cada item en el conjunto de entrenamiento
    item_popularity = defaultdict(int)
    for uid, iid, rating in trainset.all_ratings():
        item_popularity[iid] += rating

    #Se ordenan los items por popularidad
    popular_items = sorted(item_popularity.items(), key=lambda x: x[1], reverse=True)

    #Diccionario para las recomendaciones
    top_n = defaultdict(list)

    #Para cada usuario, se recomiendan los n items con mayor socre de popularidad y que no haya visto
    for uid in trainset.all_users():
        user_items = set(iid for (iid, _) in trainset.ur[uid])
        count = 0
        for iid, _ in popular_items:
            if iid not in user_items:
                top_n[trainset.to_raw_uid(uid)].append((trainset.to_raw_iid(iid), item_popularity[iid]))
                count += 1
                if count >= n:
                    break

    return top_n

Se obtiene la lista de recomendación top 10 para cada usuario y se muestra como ejemplo el caso del con ID = 457. En la lista de recomendación, el primer elemento de cada tupla es la ID del ítem, y el segundo es su score de popularidad:

In [41]:
top_n_most_popular = get_top_n_most_popular(trainset, n=10)
print(top_n_most_popular["457"])

[('28', 2962.4600000000028), ('7', 2953.4699999999953), ('39', 2933.200000000002), ('32', 2924.2300000000014), ('0', 2917.600000000006), ('21', 2880.129999999997), ('38', 2877.4099999999994), ('8', 2873.2600000000034), ('2', 2865.4000000000024), ('22', 2842.2800000000043)]


Se muestran los títulos de las películas recomendadas al usuario ID = 457 junto a su score de popularidad:

In [ ]:
for item_id, score in top_n_most_popular["457"]:
    print(f"{info_pelis[int(item_id)]}: {score:.8f}")

28: 2962.46000000
7: 2953.47000000
39: 2933.20000000
32: 2924.23000000
0: 2917.60000000
21: 2880.13000000
38: 2877.41000000
8: 2873.26000000
2: 2865.40000000
22: 2842.28000000


## 5- Random

Se realiza el procedimiento para recomendar a cada usuario 10 videojuegos aleatorios que no haya visto. Se define una función que realiza la recomendación top N para cada usuario, utilizando una seed de random para que sea replicable:

In [ ]:
import random

def get_top_n_random(trainset, n=10):
    random.seed(42)
    all_items = set(iid for iid in range(trainset.n_items))
    top_n = defaultdict(list)

    for uid in trainset.all_users():
        user_items = set(iid for (iid, _) in trainset.ur[uid])
        available_items = list(all_items - user_items)
        random_items = random.sample(available_items, min(n, len(available_items)))
        top_n[trainset.to_raw_uid(uid)] = [(trainset.to_raw_iid(iid)) for iid in random_items]

    return top_n

Se obtiene la lista de recomendación top 10 para cada usuario y se muestra como ejemplo el caso del con ID = 457:

In [90]:
top_n_random = get_top_n_random(trainset, n=10)
print(top_n_random["457"])

['38', '35', '17', '12', '39', '22', '6', '0', '27', '3']


Se muestran los títulos de las películas recomendadas al usuario ID = 457:

In [ ]:
for item_id in top_n_random["457"]:
    print(info_pelis[int(item_id)])